# RL Zoo for Campus Load Shedding — Beating the Greedy Baseline

**Companion notebook to the IEEE NIGERCON 2026 paper.**

The original benchmark found that a one-step **Greedy** heuristic outperformed
PPO by 14.8 %. This notebook tries hard to close (or invert) that gap by
running a *zoo* of reinforcement-learning algorithms and environment
variants, including several tricks specifically designed to match or beat
Greedy:

1. **Behavioural-cloning warm-start** — pre-train the PPO policy to imitate
   Greedy, then fine-tune. Should match Greedy at minimum.
2. **Action-masked PPO** — mask actions that violate the critical-cap
   constraints.
3. **Continuous-control SAC / TD3** — smaller effective action space (3 floats
   instead of 216 discrete actions).
4. **Recurrent PPO** — captures the diurnal demand cycle in policy memory.
5. **Curriculum learning** — start the agent in Normal/Minor deficit and
   ramp up to Emergency as training progresses.
6. **Hybrid PPO + Greedy ensemble** — Greedy in deep deficits where it is
   provably near-optimal, PPO where temporal credit assignment helps.

Throughout the notebook **every plot is a standalone matplotlib figure** —
no `subplots`, no `axes[i, j]`. This makes individual figures easy to lift
into slides or the paper.

> **Runtime budget knob.** Most experiments use a modest `BASE_TIMESTEPS`
> default so the whole notebook completes in well under an hour on a
> commodity GPU. Raise `BASE_TIMESTEPS` to 200 000+ for paper-grade results.

## 1. Install dependencies

In [ ]:
import sys, subprocess

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])

for pkg in ["stable-baselines3[extra]", "sb3-contrib", "gymnasium", "torch"]:
    try:
        __import__(pkg.split("[")[0].replace("-", "_"))
    except ImportError:
        print(f"installing {pkg}…")
        pip_install(pkg)

## 2. Imports and reproducibility

In [ ]:
import os, random, time, json, pickle, warnings
from pathlib import Path
from dataclasses import dataclass, field, asdict
from collections import defaultdict, deque
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import scipy.stats as st
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces

from stable_baselines3 import PPO, DQN, A2C, SAC, TD3
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

# sb3-contrib is optional - skip the algos we cannot import
try:
    from sb3_contrib import QRDQN, TRPO, RecurrentPPO, MaskablePPO
    from sb3_contrib.common.maskable.utils import get_action_masks
    SB3_CONTRIB = True
except Exception as e:
    print(f"sb3-contrib unavailable ({e}); skipping QR-DQN / TRPO / RecurrentPPO / MaskablePPO")
    SB3_CONTRIB = False

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
mpl.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.size": 11,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("torch:", torch.__version__, "| gym:", gym.__version__)
print("CUDA available:", torch.cuda.is_available())

## 3. Global configuration

A single `Config` dataclass exposes every hyper-parameter. `BASE_TIMESTEPS`
is the master compute knob: every algorithm trains for some multiple of it.

In [ ]:
@dataclass
class Config:
    DATA_PATH:      str = "final.csv"
    SAVE_PATH:      str = "./rl_zoo_artifacts"
    FIGURE_PATH:    str = "./rl_zoo_artifacts/figures"
    MODEL_PATH:     str = "./rl_zoo_artifacts/models"
    METRIC_PATH:    str = "./rl_zoo_artifacts/metrics"

    RANDOM_SEED:    int = 42

    EPISODE_LENGTH: int = 168           # one week
    N_CLUSTERS:     int = 3
    N_CURTAIL_LEVELS: int = 6           # {0, .10, .20, .30, .40, .50}

    BASE_TIMESTEPS: int = 30_000        # master compute knob
    LONG_TIMESTEPS: int = 60_000        # for top-tier candidates
    EVAL_EPISODES:  int = 60

    # default reward weights (tuned in the paper)
    W_ENERGY:     float = 1.0
    W_CRITICAL:   float = 2.0
    W_CURTAIL:    float = 0.5
    W_REPEAT:     float = 0.2
    W_FAIRNESS:   float = 0.5

CFG = Config()
for p in [CFG.SAVE_PATH, CFG.FIGURE_PATH, CFG.MODEL_PATH, CFG.METRIC_PATH]:
    Path(p).mkdir(parents=True, exist_ok=True)

def set_seeds(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(CFG.RANDOM_SEED)

## 4. Load (or synthesise) the campus demand dataset

If `final.csv` is in the working directory we use it. Otherwise we synthesise
4 344 h × 55 buildings of demand that reproduces the diurnal + weekly +
bimodal structure of the real University of Lagos record.

In [ ]:
ORIGINAL_BUILDINGS = [
    "Feeder1_High Rise Block A", "Feeder1_High Rise Auxiliary",
    "Feeder1_High Rise Block B", "Feeder1_High Rise Block C",
    "Feeder1_College of Medicine Quarters", "Feeder1_Professor Quarters",
    "Feeder1_Alvan Ikoku", "Feeder1_Agric Ode", "Feeder1_Ozolua Road",
    "Feeder2_Computer Center", "Feeder2_Health Centre", "Feeder2_Moremi Hall",
    "Feeder2_Kosoko Drive", "Feeder2_Tinubu Close", "Feeder2_CITS Market",
    "Feeder3_Engineering", "Feeder3_HV Laboratory",
    "Feeder3_Auditorium Chiller", "Feeder3_Arts Block South",
    "Feeder4_Main Library", "Feeder4_Conference Center",
    "Feeder4_Business Admin", "Feeder4_Senate House",
    "Feeder5_Faculty of Sciences", "Feeder5_Chemical Engineering Building",
    "Feeder5_Jaja Hall of Residence", "Feeder5_Mariere Hall of Residence",
    "Feeder5_Arts Theatre", "Feeder5_Main Auditorium",
    "Feeder5_Jaja Shopping Complex", "Feeder5_Mariere Shopping Complex",
    "Feeder6_Hydraulic Research Center", "Feeder6_Guest House",
    "Feeder6_Tayo Aderinokun Hall", "Feeder6_Advanced Legal Studies",
    "Feeder6_Faculty Law", "Feeder6_Faculty of Arts Blocks",
    "Feeder7_Access Bank", "Feeder7_Works & Physical Planning Building",
    "Feeder7_Faculty of Social Sciences", "Feeder7_Distance Learning Institute",
    "Feeder7_Honours Hostel", "Feeder7_International School University of Lagos",
    "Feeder7_Unilag Consults", "Feeder7_Ransome Kuti Quarters",
    "Feeder7_Henry Carr Post Graduate Hall of Residence",
    "Feeder7_Biobaku Hall of Residence", "Feeder8_New Hall Hostels",
    "Feeder8_New Hall Complex", "Feeder8_Main Mosque", "Feeder8_Chapel-1",
    "Feeder8_Chapel-2", "Feeder8_Sports Centre",
    "Feeder8_Faculty of Environmental Sciences", "Feeder8_Multipurpose Hall",
]
assert len(ORIGINAL_BUILDINGS) == 55

def synthesise_demand(n_hours=4344, seed=42):
    '''Synthesise a campus demand record with realistic structure.'''
    rng = np.random.default_rng(seed)
    ts = pd.date_range("2024-01-01", periods=n_hours, freq="h")
    hour = ts.hour.values
    dow  = ts.dayofweek.values
    # Diurnal: low at night, peak around 12-13h
    diurnal = 0.3 + 0.7 * np.exp(-((hour - 12.5) ** 2) / 25)
    # Weekly: ~70 % on weekends
    weekly = np.where(dow >= 5, 0.70, 1.0)
    cyclic = diurnal * weekly                          # [n_hours]

    data = {"Timestamp": ts}
    for b in ORIGINAL_BUILDINGS:
        base       = rng.uniform(1.5, 8.0)
        peak       = rng.uniform(2.0, 6.0)
        noise      = rng.normal(0, 0.4, n_hours)
        anomalies  = rng.normal(0, 0.05, n_hours) * (rng.random(n_hours) > 0.99) * 10
        d = base + peak * cyclic + noise + anomalies
        data[b] = np.maximum(d, 0.3)
    return pd.DataFrame(data)

if Path(CFG.DATA_PATH).exists():
    print(f"Using {CFG.DATA_PATH}")
    df = pd.read_csv(CFG.DATA_PATH)
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])
else:
    print(f"{CFG.DATA_PATH} not found — synthesising demand.")
    df = synthesise_demand(seed=CFG.RANDOM_SEED)

df = df.sort_values("Timestamp").reset_index(drop=True)
building_cols = [c for c in df.columns if c != "Timestamp"]
total_demand = df[building_cols].sum(axis=1)

print(f"Shape       : {df.shape}")
print(f"Buildings   : {len(building_cols)}")
print(f"Span        : {df['Timestamp'].min()} → {df['Timestamp'].max()}")
print(f"Mean demand : {total_demand.mean():.1f} kW")

## 5. Exploratory data analysis (one plot per figure)

The original paper used a 2×2 subplot grid. Here every panel becomes its own
matplotlib figure, easier to drop into slides or supplementary material.

### 5.1 Total campus demand, full record

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(df["Timestamp"], total_demand, lw=0.6, color="steelblue")
plt.title("Total campus demand — full record")
plt.xlabel("Time"); plt.ylabel("Demand (kW)")
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p01_total_demand_full.png")
plt.show()

### 5.2 First week — hourly detail

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(df["Timestamp"].iloc[:168], total_demand.iloc[:168],
         color="darkorange", lw=1.2)
plt.title("First week — hourly detail")
plt.xlabel("Time"); plt.ylabel("Demand (kW)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p02_first_week.png")
plt.show()

### 5.3 Distribution of hourly demand

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.hist(total_demand, bins=50, color="seagreen", edgecolor="white")
plt.title("Distribution of hourly demand")
plt.xlabel("Demand (kW)"); plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p03_demand_hist.png")
plt.show()

### 5.4 Average diurnal profile

In [ ]:
hourly = pd.DataFrame({"hour": df["Timestamp"].dt.hour, "D": total_demand}) \
            .groupby("hour")["D"]
m, s = hourly.mean(), hourly.std()
plt.figure(figsize=(8, 4.5))
plt.plot(m.index, m.values, color="crimson", lw=2, label="mean")
plt.fill_between(m.index, m - s, m + s, alpha=0.2, color="crimson", label="±1σ")
plt.title("Average diurnal profile")
plt.xlabel("Hour of day"); plt.ylabel("Demand (kW)")
plt.legend()
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p04_diurnal.png")
plt.show()

## 6. Building clustering and priority table

We carry the expert cluster assignments forward from the paper unchanged.

In [ ]:
CLUSTER_MAP = {
    "Feeder1_High Rise Block A": 2, "Feeder1_High Rise Auxiliary": 2,
    "Feeder1_High Rise Block B": 2, "Feeder1_High Rise Block C": 2,
    "Feeder1_College of Medicine Quarters": 2, "Feeder1_Professor Quarters": 2,
    "Feeder1_Alvan Ikoku": 2, "Feeder1_Agric Ode": 1, "Feeder1_Ozolua Road": 2,
    "Feeder2_Computer Center": 0, "Feeder2_Health Centre": 0,
    "Feeder2_Moremi Hall": 2, "Feeder2_Kosoko Drive": 2, "Feeder2_Tinubu Close": 2,
    "Feeder2_CITS Market": 2, "Feeder3_Engineering": 0, "Feeder3_HV Laboratory": 0,
    "Feeder3_Auditorium Chiller": 1, "Feeder3_Arts Block South": 1,
    "Feeder4_Main Library": 0, "Feeder4_Conference Center": 1,
    "Feeder4_Business Admin": 1, "Feeder4_Senate House": 0,
    "Feeder5_Faculty of Sciences": 1, "Feeder5_Chemical Engineering Building": 1,
    "Feeder5_Jaja Hall of Residence": 2, "Feeder5_Mariere Hall of Residence": 2,
    "Feeder5_Arts Theatre": 1, "Feeder5_Main Auditorium": 1,
    "Feeder5_Jaja Shopping Complex": 2, "Feeder5_Mariere Shopping Complex": 2,
    "Feeder6_Hydraulic Research Center": 1, "Feeder6_Guest House": 2,
    "Feeder6_Tayo Aderinokun Hall": 2, "Feeder6_Advanced Legal Studies": 1,
    "Feeder6_Faculty Law": 1, "Feeder6_Faculty of Arts Blocks": 1,
    "Feeder7_Access Bank": 2, "Feeder7_Works & Physical Planning Building": 0,
    "Feeder7_Faculty of Social Sciences": 1,
    "Feeder7_Distance Learning Institute": 1, "Feeder7_Honours Hostel": 2,
    "Feeder7_International School University of Lagos": 1,
    "Feeder7_Unilag Consults": 1, "Feeder7_Ransome Kuti Quarters": 2,
    "Feeder7_Henry Carr Post Graduate Hall of Residence": 2,
    "Feeder7_Biobaku Hall of Residence": 2, "Feeder8_New Hall Hostels": 2,
    "Feeder8_New Hall Complex": 2, "Feeder8_Main Mosque": 2,
    "Feeder8_Chapel-1": 2, "Feeder8_Chapel-2": 2, "Feeder8_Sports Centre": 2,
    "Feeder8_Faculty of Environmental Sciences": 1, "Feeder8_Multipurpose Hall": 1,
}

cluster_loads = pd.DataFrame({"Timestamp": df["Timestamp"]})
for k in range(CFG.N_CLUSTERS):
    cols = [b for b in building_cols if CLUSTER_MAP[b] == k]
    cluster_loads[f"Cluster{k}"] = df[cols].sum(axis=1)
cluster_loads["TotalDemand"] = total_demand
print(cluster_loads.head(3))

### 6.1 Cluster-level demand over the first two weeks

In [ ]:
plt.figure(figsize=(11, 4.5))
colours = ["#d62728", "#1f77b4", "#2ca02c"]
for k, c in zip(range(3), colours):
    plt.plot(cluster_loads["Timestamp"].iloc[:336],
             cluster_loads[f"Cluster{k}"].iloc[:336],
             label=f"Cluster {k}", lw=1.1, color=c)
plt.title("Cluster-level demand — first two weeks")
plt.xlabel("Time"); plt.ylabel("Demand (kW)")
plt.legend(loc="upper right")
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p05_cluster_demand.png")
plt.show()

### 6.2 Priority table

In [ ]:
def priority_for(b: str):
    bl = b.lower()
    if "health centre" in bl:               return 10, True,  0.00
    if "hv laboratory" in bl:               return  9, True,  0.10
    if "computer center" in bl:             return  9, True,  0.10
    if "engineering" in bl and "chemical" not in bl: return 9, True, 0.10
    if "library" in bl:                     return  8, False, 0.20
    if "senate" in bl:                      return  8, False, 0.20
    if "works & physical" in bl:            return  8, True,  0.20
    if any(k in bl for k in ["faculty", "auditorium", "auditorium chiller"]): return 7, False, 0.30
    if any(k in bl for k in ["arts theatre", "conference", "business admin"]): return 7, False, 0.30
    if "distance learning" in bl or "advanced legal" in bl: return 7, False, 0.30
    if "international school" in bl or "unilag consults" in bl: return 7, False, 0.30
    if "hydraulic research" in bl:          return 7, False, 0.30
    if "agric ode" in bl:                   return 6, False, 0.30
    if any(k in bl for k in ["hall", "hostel", "high rise"]): return 5, False, 0.40
    if any(k in bl for k in ["quarters", "guest house", "kosoko", "tinubu", "alvan", "ozolua"]): return 5, False, 0.40
    if any(k in bl for k in ["shopping", "cits market", "access bank"]): return 3, False, 0.50
    if "mosque" in bl or "chapel" in bl:    return 2, False, 0.50
    if "sports centre" in bl or "multipurpose" in bl: return 3, False, 0.50
    return 5, False, 0.40

rows = []
for b in building_cols:
    pr, crit, mc = priority_for(b)
    rows.append({"Building": b, "Cluster": CLUSTER_MAP[b],
                 "Priority": pr, "Critical": crit, "MaxCurtailment": mc})
priority_df = pd.DataFrame(rows).sort_values("Priority", ascending=False).reset_index(drop=True)
print(priority_df.head(10))

### 6.3 Distribution of building priorities

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.hist(priority_df["Priority"], bins=range(1, 12),
         color="indigo", edgecolor="white", align="left")
plt.xticks(range(1, 11))
plt.title("Distribution of building priorities")
plt.xlabel("Priority (1–10)"); plt.ylabel("Number of buildings")
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p06_priority_hist.png")
plt.show()

### 6.4 Priority composition per cluster

In [ ]:
plt.figure(figsize=(9, 4.5))
ct = priority_df.groupby(["Cluster", "Priority"]).size().unstack(fill_value=0)
ct.plot(kind="bar", stacked=True, colormap="viridis", edgecolor="white",
        ax=plt.gca())
plt.title("Priority composition per cluster")
plt.xlabel("Cluster"); plt.ylabel("Number of buildings")
plt.legend(title="Priority", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p07_priority_by_cluster.png")
plt.show()

## 7. Synthetic power-deficit generator

Same five regimes as the paper — Normal/Minor/Moderate/Major/Emergency —
parameterised by an availability ratio.

In [ ]:
class PowerScenario:
    REG = {"Normal": (0.95, 1.05), "Minor": (0.80, 0.95),
           "Moderate": (0.60, 0.80), "Major": (0.40, 0.60),
           "Emergency": (0.20, 0.40)}
    def __init__(self, rng=None):
        self.rng = rng or np.random.default_rng(CFG.RANDOM_SEED)
    def sample(self, demand, scen="Random"):
        if scen == "Random":
            scen = self.rng.choice(list(self.REG))
        lo, hi = self.REG[scen]
        r = self.rng.uniform(lo, hi)
        return demand * r, max(0.0, 1.0 - r)

### 7.1 Available-power realisations under each regime (one week)

In [ ]:
plt.figure(figsize=(11, 5))
gen = PowerScenario()
demo = cluster_loads["TotalDemand"].iloc[:168].values
plt.plot(demo, label="Demand", color="black", lw=1.5)
cols = ["#2ca02c", "#bcbd22", "#ff7f0e", "#d62728", "#7f0000"]
for scen, col in zip(PowerScenario.REG, cols):
    avail = np.array([gen.sample(d, scen)[0] for d in demo])
    plt.plot(avail, label=scen, alpha=0.85, color=col)
plt.title("Available power under each scenario regime")
plt.xlabel("Hour"); plt.ylabel("Power (kW)")
plt.legend(ncol=3)
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p08_scenarios.png")
plt.show()

## 8. Three environment flavours

We expose the campus microgrid as three Gymnasium environments that share
the same dynamics but differ in what the agent controls:

* **`CampusEnvDiscrete`** — original 216-action discrete space, 8-D state
  + sin/cos hour encoding (10-D).
* **`CampusEnvContinuous`** — `Box([0, 0.5]³)` action: one continuous
  curtailment fraction per cluster. Smaller effective control space,
  required by SAC/TD3.
* **`CampusEnvMasked`** — same as discrete but exposes a per-step action
  mask that forbids actions violating per-cluster `MaxCurtailment`. Required
  by `MaskablePPO`.

In [ ]:
CURTAIL_LEVELS = np.array([0.0, 0.10, 0.20, 0.30, 0.40, 0.50])

def jain(x):
    x = np.asarray(x, dtype=float)
    return (x.sum() ** 2) / (len(x) * np.square(x).sum() + 1e-9)

def cyclic_hour(h):
    th = 2 * np.pi * h / 24
    return np.sin(th), np.cos(th)


class CampusEnvBase(gym.Env):
    """Shared dynamics for the three flavours."""
    metadata = {"render_modes": ["human"]}

    def __init__(self, cluster_data, priority_df, cfg=CFG,
                 scenario="Random", episode_length=None,
                 reward_weights=None, seed=None):
        super().__init__()
        self.cluster_data = cluster_data.reset_index(drop=True)
        self.priority_df = priority_df
        self.cfg = cfg
        self.scenario = scenario
        self.episode_length = episode_length or cfg.EPISODE_LENGTH

        d = dict(energy=cfg.W_ENERGY, critical=cfg.W_CRITICAL,
                 curtail=cfg.W_CURTAIL, repeat=cfg.W_REPEAT,
                 fairness=cfg.W_FAIRNESS)
        if reward_weights: d.update(reward_weights)
        self.w = d

        self._crit_share = np.zeros(cfg.N_CLUSTERS, dtype=np.float32)
        self._max_cap = np.zeros(cfg.N_CLUSTERS, dtype=np.float32)
        for k in range(cfg.N_CLUSTERS):
            sub = priority_df[priority_df["Cluster"] == k]
            crit_mass = sub.loc[sub["Critical"], "Priority"].sum()
            total = sub["Priority"].sum() + 1e-9
            self._crit_share[k] = crit_mass / total
            # most permissive per-cluster curtailment cap
            self._max_cap[k] = sub["MaxCurtailment"].max() if len(sub) else 0.5

        # 10-D obs: 3 cluster demands, total, avail, deficit, sin(h), cos(h),
        # prev action mean, prev deficit
        self.observation_space = spaces.Box(low=-1.0, high=2.0,
                                            shape=(10,), dtype=np.float32)

        self._demand_max = float(self.cluster_data["TotalDemand"].max())
        self._cluster_max = {k: float(self.cluster_data[f"Cluster{k}"].max())
                              for k in range(cfg.N_CLUSTERS)}

        self._scen_gen = PowerScenario(np.random.default_rng(seed or cfg.RANDOM_SEED))
        self._rng = np.random.default_rng(seed or cfg.RANDOM_SEED)

    # ------------------ helpers shared by subclasses -----------------------
    def _hour_of_day(self):
        return int(self.cluster_data["Timestamp"].iloc[self._cursor].hour)

    def _make_obs(self, prev_action_mean=0.0, prev_deficit=0.0):
        row = self.cluster_data.iloc[self._cursor]
        D = np.array([row[f"Cluster{k}"] for k in range(self.cfg.N_CLUSTERS)],
                     dtype=np.float32)
        D_tot = float(row["TotalDemand"])
        avail, deficit = self._scen_gen.sample(D_tot, self.scenario)
        self._last_avail, self._last_deficit = avail, deficit
        sh, ch = cyclic_hour(self._hour_of_day())
        obs = np.array([
            D[0] / (self._cluster_max[0] + 1e-9),
            D[1] / (self._cluster_max[1] + 1e-9),
            D[2] / (self._cluster_max[2] + 1e-9),
            D_tot / (self._demand_max + 1e-9),
            avail / (self._demand_max + 1e-9),
            deficit,
            (sh + 1) / 2,
            (ch + 1) / 2,
            prev_action_mean,
            prev_deficit,
        ], dtype=np.float32)
        return obs, D, D_tot, avail, deficit

    def _apply_curtailment(self, curtailment, D, D_tot, avail):
        # cap to per-cluster max
        curtailment = np.minimum(curtailment, self._max_cap)
        target_served = (1.0 - curtailment) * D
        # if still over supply, scale all clusters
        if target_served.sum() > avail and target_served.sum() > 1e-6:
            scale = avail / target_served.sum()
            served = target_served * scale
        else:
            served = target_served
        curtailed = D - served
        return served, curtailed, curtailment

    def _step_reward(self, served, curtailed, D, D_tot, action_mean, prev_action_mean):
        served_ratio = served.sum() / (D_tot + 1e-9)
        critical_pen = (self._crit_share * curtailed).sum() / (D_tot + 1e-9)
        curtail_pen = curtailed.sum() / (D_tot + 1e-9)
        repeat_pen = 1.0 if abs(action_mean - prev_action_mean) < 1e-6 else 0.0
        served_ratios = served / np.maximum(D, 1e-6)
        fairness = jain(served_ratios)
        r = (self.w["energy"]   * served_ratio
             - self.w["critical"] * critical_pen
             - self.w["curtail"]  * curtail_pen
             - self.w["repeat"]   * repeat_pen
             + self.w["fairness"] * fairness)
        return float(r), dict(energy=self.w["energy"]*served_ratio,
                              critical=-self.w["critical"]*critical_pen,
                              curtail=-self.w["curtail"]*curtail_pen,
                              repeat=-self.w["repeat"]*repeat_pen,
                              fairness=self.w["fairness"]*fairness)

    def reset(self, seed=None, options=None):
        if seed is not None:
            self._rng = np.random.default_rng(seed)
            self._scen_gen = PowerScenario(np.random.default_rng(seed))
        max_start = len(self.cluster_data) - self.episode_length - 1
        self._cursor = int(self._rng.integers(0, max_start))
        self._t = 0
        self._prev_action_mean = 0.0
        self._prev_deficit = 0.0
        self._history = []
        obs, *_ = self._make_obs()
        return obs, {}


class CampusEnvDiscrete(CampusEnvBase):
    """216-action discrete control."""
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self.action_space = spaces.Discrete(
            self.cfg.N_CURTAIL_LEVELS ** self.cfg.N_CLUSTERS)

    def decode_action(self, a):
        L = self.cfg.N_CURTAIL_LEVELS
        idx = [(a // (L ** k)) % L for k in range(self.cfg.N_CLUSTERS)]
        return CURTAIL_LEVELS[idx]

    def step(self, action):
        action = int(action)
        curt = self.decode_action(action)
        obs, D, D_tot, avail, deficit = self._make_obs(
            self._prev_action_mean, self._prev_deficit)
        served, curtailed, curt_capped = self._apply_curtailment(curt, D, D_tot, avail)
        action_mean = float(np.mean(curt_capped))
        r, br = self._step_reward(served, curtailed, D, D_tot,
                                  action_mean, self._prev_action_mean)
        self._history.append(dict(demand=D, served=served, curtailed=curtailed,
                                  available=avail, deficit=deficit,
                                  action=action, breakdown=br))
        self._prev_action_mean, self._prev_deficit = action_mean, deficit
        self._cursor += 1
        self._t += 1
        terminated = self._t >= self.episode_length
        info = dict(served=float(served.sum()), curtailed=float(curtailed.sum()),
                    breakdown=br, available=avail, deficit=deficit)
        return obs, r, terminated, False, info


class CampusEnvContinuous(CampusEnvBase):
    """Box([0, 0.5]^3) continuous control for SAC / TD3."""
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self.action_space = spaces.Box(low=0.0, high=0.5,
                                       shape=(self.cfg.N_CLUSTERS,),
                                       dtype=np.float32)

    def step(self, action):
        curt = np.clip(action.astype(np.float32), 0.0, 0.5)
        obs, D, D_tot, avail, deficit = self._make_obs(
            self._prev_action_mean, self._prev_deficit)
        served, curtailed, curt_capped = self._apply_curtailment(curt, D, D_tot, avail)
        action_mean = float(curt_capped.mean())
        r, br = self._step_reward(served, curtailed, D, D_tot,
                                  action_mean, self._prev_action_mean)
        self._history.append(dict(demand=D, served=served, curtailed=curtailed,
                                  available=avail, deficit=deficit,
                                  action=curt_capped, breakdown=br))
        self._prev_action_mean, self._prev_deficit = action_mean, deficit
        self._cursor += 1
        self._t += 1
        terminated = self._t >= self.episode_length
        info = dict(served=float(served.sum()), curtailed=float(curtailed.sum()),
                    breakdown=br, available=avail, deficit=deficit)
        return obs, r, terminated, False, info


class CampusEnvMasked(CampusEnvDiscrete):
    """Same as discrete but exposes per-step action_masks() for MaskablePPO."""
    def action_masks(self):
        # Allowed actions = those whose decoded curtailment ≤ per-cluster cap
        L = self.cfg.N_CURTAIL_LEVELS
        n = L ** self.cfg.N_CLUSTERS
        mask = np.ones(n, dtype=bool)
        for a in range(n):
            idx = [(a // (L ** k)) % L for k in range(self.cfg.N_CLUSTERS)]
            curt = CURTAIL_LEVELS[idx]
            if np.any(curt > self._max_cap + 1e-6):
                mask[a] = False
        return mask

## 9. Heuristic baselines and the evaluation harness

These are the controllers the RL agents have to beat — especially **Greedy**.

In [ ]:
class BaseController:
    name = "Base"
    def __init__(self, env): self.env = env
    def select_action(self, obs, info=None): raise NotImplementedError

    def evaluate(self, n_episodes=100, seed=0):
        rewards, served, curtail, critical, fair, runtime = [], [], [], [], [], []
        for ep in range(n_episodes):
            obs, _ = self.env.reset(seed=seed + ep)
            r_tot = s_tot = c_tot = cr_tot = 0.0
            t0 = time.perf_counter()
            done = False
            while not done:
                a = self.select_action(obs)
                obs, r, done, _, info = self.env.step(a)
                r_tot  += r
                s_tot  += info["served"]
                c_tot  += info["curtailed"]
                cr_tot += info["breakdown"]["critical"]
            runtime.append(time.perf_counter() - t0)
            served_by = np.sum([h["served"] for h in self.env._history], axis=0)
            demand_by = np.sum([h["demand"] for h in self.env._history], axis=0)
            ratios = served_by / np.maximum(demand_by, 1e-9)
            rewards.append(r_tot); served.append(s_tot); curtail.append(c_tot)
            critical.append(cr_tot); fair.append(jain(ratios))
        return dict(reward=np.array(rewards), served=np.array(served),
                    curtail=np.array(curtail), critical=np.array(critical),
                    fairness=np.array(fair), runtime=np.array(runtime))


# ---------------- baselines (discrete env) ---------------------------------
class RandomCtrl(BaseController):
    name = "Random"
    def select_action(self, obs, info=None):
        return self.env.action_space.sample()

class RoundRobinCtrl(BaseController):
    name = "RoundRobin"
    def __init__(self, env): super().__init__(env); self._c = 0
    def select_action(self, obs, info=None):
        a = self._c % self.env.action_space.n
        self._c += 1
        return a

class StaticPriorityCtrl(BaseController):
    """Curtail Cluster 2 hard, Cluster 1 mildly, Cluster 0 never."""
    name = "StaticPriority"
    def select_action(self, obs, info=None):
        idx = [0, 2, 4]            # 0%, 20%, 40%
        L = self.env.cfg.N_CURTAIL_LEVELS
        return int(idx[0] + idx[1] * L + idx[2] * L ** 2)

class LowestPriorityFirstCtrl(BaseController):
    name = "LowestPriorityFirst"
    def select_action(self, obs, info=None):
        deficit = obs[5]
        L = self.env.cfg.N_CURTAIL_LEVELS
        if deficit < 0.05:
            return int(0)
        # curtail cluster 2 proportional to deficit, then cluster 1
        lvl2 = int(min(L - 1, round(deficit * (L - 1))))
        lvl1 = int(min(L - 1, round(deficit * (L - 1) * 0.6)))
        return int(0 + lvl1 * L + lvl2 * L ** 2)

class GreedyCtrl(BaseController):
    """Match supply to demand by curtailing smallest cluster first."""
    name = "Greedy"
    def select_action(self, obs, info=None):
        e = self.env
        cursor = e._cursor
        row = e.cluster_data.iloc[cursor]
        D = np.array([row[f"Cluster{k}"] for k in range(e.cfg.N_CLUSTERS)])
        D_tot = D.sum()
        avail = e._last_avail if hasattr(e, "_last_avail") else D_tot
        if avail >= D_tot:
            return 0                                # no curtailment
        deficit_kw = D_tot - avail
        # curtail starting from lowest-priority cluster (2 → 1 → 0)
        L = e.cfg.N_CURTAIL_LEVELS
        idx = [0, 0, 0]
        remaining = deficit_kw
        for k in [2, 1, 0]:
            if remaining <= 0:
                break
            # what curtail level brings cluster k down by `remaining`?
            need_frac = min(0.5, remaining / max(D[k], 1e-6))
            lvl = int(round(need_frac * (L - 1) / 0.5))
            lvl = min(lvl, L - 1)
            idx[k] = lvl
            remaining -= CURTAIL_LEVELS[lvl] * D[k]
        return int(idx[0] + idx[1] * L + idx[2] * L ** 2)

### 9.1 Baseline reference results (one quick run)

In [ ]:
ref_env = CampusEnvDiscrete(cluster_loads, priority_df, seed=CFG.RANDOM_SEED + 99)
ref_results = {}
for cls in [RandomCtrl, RoundRobinCtrl, StaticPriorityCtrl,
            LowestPriorityFirstCtrl, GreedyCtrl]:
    ctrl = cls(ref_env)
    m = ctrl.evaluate(n_episodes=CFG.EVAL_EPISODES,
                      seed=CFG.RANDOM_SEED + 12345)
    ref_results[ctrl.name] = m
    print(f"{ctrl.name:25s}  reward μ = {m['reward'].mean():8.3f} ± {m['reward'].std():.3f}")

## 10. RL training engine

A `TrainingLogger` callback captures episode rewards so we can plot a
learning curve per algorithm.  All training and evaluation goes through a
helper so the experiments stay comparable.

In [ ]:
class TrainingLogger(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.episode_rewards = []
    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self.episode_rewards.append(float(info["episode"]["r"]))
        return True


def make_env_fn(env_cls, scenario="Random", reward_weights=None, seed=0):
    def _thunk():
        env = env_cls(cluster_loads, priority_df, scenario=scenario,
                      reward_weights=reward_weights, seed=seed)
        return Monitor(env)
    return _thunk


def train_and_log(name, algo_factory, env_cls=CampusEnvDiscrete,
                  total_timesteps=None, env_kwargs=None):
    '''Train an SB3-compatible model, return (model, logger).'''
    total_timesteps = total_timesteps or CFG.BASE_TIMESTEPS
    env_kwargs = env_kwargs or {}
    set_seeds(CFG.RANDOM_SEED)
    venv = DummyVecEnv([make_env_fn(env_cls, seed=CFG.RANDOM_SEED, **env_kwargs)])
    logger = TrainingLogger()
    model = algo_factory(venv)
    print(f"Training {name} for {total_timesteps:,} steps…")
    t0 = time.perf_counter()
    model.learn(total_timesteps=total_timesteps, callback=logger,
                progress_bar=False)
    dt = time.perf_counter() - t0
    print(f"{name} done in {dt:.1f}s — final 20-ep mean reward: "
          f"{np.mean(logger.episode_rewards[-20:]) if logger.episode_rewards else float('nan'):.2f}")
    return model, logger


def smoothed(x, k=20):
    if len(x) == 0: return np.array([])
    k = min(k, len(x))
    return np.convolve(x, np.ones(k) / k, mode="valid")


def plot_curve(logger, name, colour="steelblue"):
    plt.figure(figsize=(9, 4.4))
    r = logger.episode_rewards
    if r:
        plt.plot(r, alpha=0.35, color=colour, label="episode reward")
        sm = smoothed(r, 20)
        plt.plot(np.arange(len(r) - len(sm) + 1, len(r) + 1),
                 sm, color="darkred", lw=2, label="20-ep moving avg")
    plt.title(f"{name} training curve")
    plt.xlabel("Episode"); plt.ylabel("Reward")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{CFG.FIGURE_PATH}/curve_{name.replace(' ', '_')}.png")
    plt.show()


# global registries so every later cell can reach the trained models
TRAINED = {}     # name -> stable-baselines3 model
LOGGERS = {}     # name -> TrainingLogger

## 11. RL algorithm zoo

We train **eleven RL variants**.  Each gets its own training cell and its
own standalone learning-curve plot.

| # | Name | Algorithm family | Trick |
|---|------|------------------|-------|
| 1 | `PPO`              | Policy gradient (on-policy) | Vanilla baseline |
| 2 | `PPO-Large`        | Policy gradient | Wider MLP (256-256) |
| 3 | `PPO-Curriculum`   | Policy gradient | Easier scenarios first, harder later |
| 4 | `PPO-Tuned`        | Policy gradient | Larger batch, lower LR, more epochs |
| 5 | `A2C`              | Actor-critic | Synchronous baseline |
| 6 | `DQN`              | Value-based (off-policy) | Vanilla |
| 7 | `QR-DQN`           | Distributional Q | sb3-contrib |
| 8 | `RecurrentPPO`     | Policy gradient + LSTM | Captures diurnal cycle |
| 9 | `MaskablePPO`      | Policy gradient + action mask | Forbids capacity-violating actions |
| 10 | `TRPO`            | Trust-region policy gradient | sb3-contrib |
| 11 | `SAC`             | Maximum-entropy actor-critic | **Continuous action space** |
| 12 | `TD3`             | Deterministic actor-critic | **Continuous action space** |
| 13 | `Hybrid-PPO+Greedy` | Ensemble | Greedy in deep deficit, PPO otherwise |

The reward shaping, hyper-parameters and seeds are all unchanged across
variants unless explicitly noted.

### 11.1 PPO — vanilla

In [ ]:
def ppo_factory(venv):
    return PPO("MlpPolicy", venv,
               learning_rate=3e-4, n_steps=2048, batch_size=64,
               gamma=0.99, gae_lambda=0.95, ent_coef=0.01,
               clip_range=0.2, verbose=0, seed=CFG.RANDOM_SEED)
m, lg = train_and_log("PPO", ppo_factory)
TRAINED["PPO"] = m; LOGGERS["PPO"] = lg
plot_curve(lg, "PPO", "steelblue")

### 11.2 PPO-Large (wider MLP)

In [ ]:
def ppo_large_factory(venv):
    return PPO("MlpPolicy", venv,
               policy_kwargs=dict(net_arch=[256, 256]),
               learning_rate=3e-4, n_steps=2048, batch_size=64,
               gamma=0.99, gae_lambda=0.95, ent_coef=0.01,
               clip_range=0.2, verbose=0, seed=CFG.RANDOM_SEED)
m, lg = train_and_log("PPO-Large", ppo_large_factory)
TRAINED["PPO-Large"] = m; LOGGERS["PPO-Large"] = lg
plot_curve(lg, "PPO-Large", "navy")

### 11.3 PPO-Curriculum

Train in three stages: Normal → Moderate → Random.  The intuition is that
the policy first locks in the easy "no shedding" pattern, then learns to
handle deficits incrementally.

In [ ]:
def make_curriculum_env(scenario, seed):
    def _thunk():
        return Monitor(CampusEnvDiscrete(cluster_loads, priority_df,
                                         scenario=scenario, seed=seed))
    return _thunk

set_seeds(CFG.RANDOM_SEED)
curr_logger = TrainingLogger()
venv = DummyVecEnv([make_curriculum_env("Normal", CFG.RANDOM_SEED)])
ppo_curr = PPO("MlpPolicy", venv, learning_rate=3e-4, n_steps=2048,
               batch_size=64, gamma=0.99, gae_lambda=0.95, ent_coef=0.01,
               clip_range=0.2, verbose=0, seed=CFG.RANDOM_SEED)

stages = [("Normal", CFG.BASE_TIMESTEPS // 3),
          ("Moderate", CFG.BASE_TIMESTEPS // 3),
          ("Random",   CFG.BASE_TIMESTEPS // 3)]
for scen, budget in stages:
    print(f"  curriculum stage {scen}: {budget} steps")
    venv = DummyVecEnv([make_curriculum_env(scen, CFG.RANDOM_SEED)])
    ppo_curr.set_env(venv)
    ppo_curr.learn(total_timesteps=budget, callback=curr_logger, progress_bar=False)

TRAINED["PPO-Curriculum"] = ppo_curr; LOGGERS["PPO-Curriculum"] = curr_logger
plot_curve(curr_logger, "PPO-Curriculum", "purple")

### 11.4 PPO-Tuned (longer rollouts, lower LR, more epochs)

In [ ]:
def ppo_tuned_factory(venv):
    return PPO("MlpPolicy", venv,
               learning_rate=1e-4, n_steps=4096, batch_size=128,
               n_epochs=20, gamma=0.99, gae_lambda=0.95, ent_coef=0.005,
               clip_range=0.2, verbose=0, seed=CFG.RANDOM_SEED)
m, lg = train_and_log("PPO-Tuned", ppo_tuned_factory,
                      total_timesteps=CFG.LONG_TIMESTEPS)
TRAINED["PPO-Tuned"] = m; LOGGERS["PPO-Tuned"] = lg
plot_curve(lg, "PPO-Tuned", "teal")

### 11.5 PPO with Behavioural-Cloning warm-start

Strategy with the best chance of matching Greedy: pre-train the policy network
to imitate Greedy's choices on rollouts from the real environment, then hand
the warm-started policy over to PPO for fine-tuning.

In [ ]:
def collect_greedy_dataset(n_episodes=20, seed=0):
    env = CampusEnvDiscrete(cluster_loads, priority_df, seed=seed)
    greedy = GreedyCtrl(env)
    X, y = [], []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        done = False
        while not done:
            a = greedy.select_action(obs)
            X.append(obs.copy()); y.append(int(a))
            obs, _, done, _, _ = env.step(a)
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.int64)

print("Collecting expert (Greedy) trajectories…")
Xe, ye = collect_greedy_dataset(n_episodes=25, seed=CFG.RANDOM_SEED)
print(f"  collected {len(Xe)} transitions across 25 episodes")

# Build a PPO and warm-start its policy by behavioural cloning on Greedy
set_seeds(CFG.RANDOM_SEED)
bc_venv = DummyVecEnv([make_env_fn(CampusEnvDiscrete, seed=CFG.RANDOM_SEED)])
ppo_bc = PPO("MlpPolicy", bc_venv, learning_rate=3e-4, n_steps=2048,
             batch_size=64, gamma=0.99, gae_lambda=0.95, ent_coef=0.01,
             clip_range=0.2, verbose=0, seed=CFG.RANDOM_SEED)

device = ppo_bc.policy.device
Xt = torch.tensor(Xe, dtype=torch.float32, device=device)
yt = torch.tensor(ye, dtype=torch.long,    device=device)
opt = torch.optim.Adam(ppo_bc.policy.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
print("Behavioural-cloning the policy network for 200 epochs…")
for ep in range(200):
    idx = torch.randperm(len(Xt), device=device)[:1024]
    batch_x, batch_y = Xt[idx], yt[idx]
    dist = ppo_bc.policy.get_distribution(batch_x)
    logits = dist.distribution.logits
    loss = loss_fn(logits, batch_y)
    opt.zero_grad(); loss.backward(); opt.step()
    if ep % 40 == 0:
        with torch.no_grad():
            pred = dist.distribution.logits.argmax(-1)
            acc = (pred == batch_y).float().mean().item()
        print(f"  epoch {ep:3d}  loss={loss.item():.4f}  acc={acc:.3f}")

print("Fine-tuning warm-started PPO…")
bc_logger = TrainingLogger()
ppo_bc.learn(total_timesteps=CFG.BASE_TIMESTEPS, callback=bc_logger,
             progress_bar=False)
TRAINED["PPO-BC"] = ppo_bc; LOGGERS["PPO-BC"] = bc_logger
plot_curve(bc_logger, "PPO-BC", "darkgreen")

### 11.6 A2C — synchronous actor-critic

In [ ]:
def a2c_factory(venv):
    return A2C("MlpPolicy", venv, learning_rate=7e-4, gamma=0.99,
               ent_coef=0.01, verbose=0, seed=CFG.RANDOM_SEED)
m, lg = train_and_log("A2C", a2c_factory)
TRAINED["A2C"] = m; LOGGERS["A2C"] = lg
plot_curve(lg, "A2C", "orange")

### 11.7 DQN — vanilla

In [ ]:
def dqn_factory(venv):
    return DQN("MlpPolicy", venv, learning_rate=1e-4,
               buffer_size=100_000, batch_size=64, gamma=0.99,
               target_update_interval=1000, exploration_fraction=0.2,
               learning_starts=1000, verbose=0, seed=CFG.RANDOM_SEED)
m, lg = train_and_log("DQN", dqn_factory)
TRAINED["DQN"] = m; LOGGERS["DQN"] = lg
plot_curve(lg, "DQN", "firebrick")

### 11.8 QR-DQN — distributional value learning (sb3-contrib)

In [ ]:
if SB3_CONTRIB:
    def qrdqn_factory(venv):
        return QRDQN("MlpPolicy", venv, learning_rate=5e-5,
                     buffer_size=100_000, batch_size=64, gamma=0.99,
                     target_update_interval=1000, exploration_fraction=0.2,
                     learning_starts=1000, verbose=0, seed=CFG.RANDOM_SEED)
    m, lg = train_and_log("QR-DQN", qrdqn_factory)
    TRAINED["QR-DQN"] = m; LOGGERS["QR-DQN"] = lg
    plot_curve(lg, "QR-DQN", "indianred")
else:
    print("sb3-contrib not available, skipping QR-DQN")

### 11.9 Recurrent PPO — LSTM policy

In [ ]:
if SB3_CONTRIB:
    def recurrent_factory(venv):
        return RecurrentPPO("MlpLstmPolicy", venv, learning_rate=3e-4,
                            n_steps=2048, batch_size=64, gamma=0.99,
                            gae_lambda=0.95, ent_coef=0.01, clip_range=0.2,
                            verbose=0, seed=CFG.RANDOM_SEED)
    m, lg = train_and_log("RecurrentPPO", recurrent_factory)
    TRAINED["RecurrentPPO"] = m; LOGGERS["RecurrentPPO"] = lg
    plot_curve(lg, "RecurrentPPO", "darkblue")
else:
    print("sb3-contrib not available, skipping RecurrentPPO")

### 11.10 MaskablePPO — action-masked PPO

The mask forbids actions that would shed more than `MaxCurtailment` for any
cluster, so the agent never wastes exploration on infeasible joint actions.

In [ ]:
if SB3_CONTRIB:
    def masked_factory(venv):
        return MaskablePPO("MlpPolicy", venv, learning_rate=3e-4,
                           n_steps=2048, batch_size=64, gamma=0.99,
                           gae_lambda=0.95, ent_coef=0.01, clip_range=0.2,
                           verbose=0, seed=CFG.RANDOM_SEED)
    m, lg = train_and_log("MaskablePPO", masked_factory,
                          env_cls=CampusEnvMasked)
    TRAINED["MaskablePPO"] = m; LOGGERS["MaskablePPO"] = lg
    plot_curve(lg, "MaskablePPO", "magenta")
else:
    print("sb3-contrib not available, skipping MaskablePPO")

### 11.11 TRPO — trust-region policy gradient (sb3-contrib)

In [ ]:
if SB3_CONTRIB:
    def trpo_factory(venv):
        return TRPO("MlpPolicy", venv, learning_rate=1e-3, n_steps=2048,
                    batch_size=64, gamma=0.99, gae_lambda=0.95,
                    verbose=0, seed=CFG.RANDOM_SEED)
    m, lg = train_and_log("TRPO", trpo_factory)
    TRAINED["TRPO"] = m; LOGGERS["TRPO"] = lg
    plot_curve(lg, "TRPO", "olive")
else:
    print("sb3-contrib not available, skipping TRPO")

### 11.12 SAC — maximum-entropy off-policy, **continuous action space**

SAC requires a continuous action space, so we swap the environment for
`CampusEnvContinuous` whose 3-D Box action directly encodes curtailment
fractions per cluster. The effective action space is much smaller than the
216-action discrete one.

In [ ]:
def sac_factory(venv):
    return SAC("MlpPolicy", venv, learning_rate=3e-4,
               buffer_size=100_000, batch_size=128, gamma=0.99, tau=0.005,
               learning_starts=1000, verbose=0, seed=CFG.RANDOM_SEED)
m, lg = train_and_log("SAC", sac_factory,
                      env_cls=CampusEnvContinuous,
                      total_timesteps=CFG.LONG_TIMESTEPS)
TRAINED["SAC"] = m; LOGGERS["SAC"] = lg
plot_curve(lg, "SAC", "darkcyan")

### 11.13 TD3 — twin-delayed DDPG, continuous action space

In [ ]:
def td3_factory(venv):
    return TD3("MlpPolicy", venv, learning_rate=3e-4,
               buffer_size=100_000, batch_size=128, gamma=0.99,
               tau=0.005, policy_delay=2, learning_starts=1000,
               verbose=0, seed=CFG.RANDOM_SEED)
m, lg = train_and_log("TD3", td3_factory,
                      env_cls=CampusEnvContinuous,
                      total_timesteps=CFG.LONG_TIMESTEPS)
TRAINED["TD3"] = m; LOGGERS["TD3"] = lg
plot_curve(lg, "TD3", "chocolate")

### 11.14 Hybrid PPO + Greedy ensemble

A *controller of controllers*: in deep deficit (where the original paper
showed PPO collapses) we hand the wheel to Greedy; otherwise PPO drives.
Switching happens on the observed deficit ratio at each step.

In [ ]:
class HybridController:
    '''Greedy in deep deficit (>=0.4), PPO otherwise.'''
    name = "Hybrid-PPO+Greedy"
    def __init__(self, env, ppo_model, threshold=0.40):
        self.env = env
        self.ppo = ppo_model
        self.greedy = GreedyCtrl(env)
        self.thr = threshold

    def select_action(self, obs, info=None):
        deficit = float(obs[5])
        if deficit >= self.thr:
            return self.greedy.select_action(obs)
        a, _ = self.ppo.predict(obs, deterministic=True)
        return int(a)

    # reuse BaseController.evaluate verbatim
    evaluate = BaseController.evaluate

## 12. Side-by-side evaluation

Every RL variant and every baseline is evaluated on the *same* 60 paired
episodes so all numbers below are directly comparable.

In [ ]:
class RLCtrl(BaseController):
    '''Wrap any SB3-compatible model so it can plug into the eval harness.

    Handles three subtle inference differences:
    * MaskablePPO needs action_masks= passed to predict
    * RecurrentPPO needs state + episode_starts maintained across steps
    * SAC/TD3 return a numpy array (the env handles that internally)
    '''
    def __init__(self, env, model, name):
        super().__init__(env); self.model = model; self.name = name
        self._is_masked    = SB3_CONTRIB and isinstance(model, MaskablePPO)
        self._is_recurrent = SB3_CONTRIB and isinstance(model, RecurrentPPO)
        self._lstm_state = None
        self._ep_start = True

    def select_action(self, obs, info=None):
        if self._is_masked:
            masks = self.env.action_masks() if hasattr(self.env, "action_masks") else None
            a, _ = self.model.predict(obs, deterministic=True, action_masks=masks)
        elif self._is_recurrent:
            a, self._lstm_state = self.model.predict(
                obs, state=self._lstm_state,
                episode_start=np.array([self._ep_start]),
                deterministic=True)
            self._ep_start = False
        else:
            a, _ = self.model.predict(obs, deterministic=True)
        if isinstance(a, np.ndarray) and a.ndim == 0:
            return int(a)
        return a

    def evaluate(self, n_episodes=100, seed=0):
        rewards, served, curtail, critical, fair, runtime = [], [], [], [], [], []
        for ep in range(n_episodes):
            obs, _ = self.env.reset(seed=seed + ep)
            # reset recurrent state at the start of every episode
            self._lstm_state = None; self._ep_start = True
            r_tot = s_tot = c_tot = cr_tot = 0.0
            t0 = time.perf_counter()
            done = False
            while not done:
                a = self.select_action(obs)
                obs, r, done, _, info = self.env.step(a)
                r_tot  += r
                s_tot  += info["served"]
                c_tot  += info["curtailed"]
                cr_tot += info["breakdown"]["critical"]
            runtime.append(time.perf_counter() - t0)
            served_by = np.sum([h["served"] for h in self.env._history], axis=0)
            demand_by = np.sum([h["demand"] for h in self.env._history], axis=0)
            ratios = served_by / np.maximum(demand_by, 1e-9)
            rewards.append(r_tot); served.append(s_tot); curtail.append(c_tot)
            critical.append(cr_tot); fair.append(jain(ratios))
        return dict(reward=np.array(rewards), served=np.array(served),
                    curtail=np.array(curtail), critical=np.array(critical),
                    fairness=np.array(fair), runtime=np.array(runtime))

results = {}

# Baselines first - share the env
eval_env_disc = CampusEnvDiscrete(cluster_loads, priority_df,
                                  seed=CFG.RANDOM_SEED + 1001)
for cls in [RandomCtrl, RoundRobinCtrl, StaticPriorityCtrl,
            LowestPriorityFirstCtrl, GreedyCtrl]:
    ctrl = cls(eval_env_disc)
    print(f"Eval {ctrl.name}…")
    results[ctrl.name] = ctrl.evaluate(n_episodes=CFG.EVAL_EPISODES,
                                       seed=CFG.RANDOM_SEED + 12345)

# Discrete RL variants
discrete_rl_names = ["PPO", "PPO-Large", "PPO-Curriculum", "PPO-Tuned",
                     "PPO-BC", "A2C", "DQN", "QR-DQN", "RecurrentPPO",
                     "MaskablePPO", "TRPO"]
for n in discrete_rl_names:
    if n not in TRAINED: continue
    env_cls = CampusEnvMasked if n == "MaskablePPO" else CampusEnvDiscrete
    env = env_cls(cluster_loads, priority_df, seed=CFG.RANDOM_SEED + 1001)
    print(f"Eval {n}…")
    results[n] = RLCtrl(env, TRAINED[n], n).evaluate(
        n_episodes=CFG.EVAL_EPISODES, seed=CFG.RANDOM_SEED + 12345)

# Continuous RL variants
for n in ["SAC", "TD3"]:
    if n not in TRAINED: continue
    env = CampusEnvContinuous(cluster_loads, priority_df,
                              seed=CFG.RANDOM_SEED + 1001)
    print(f"Eval {n}…")
    results[n] = RLCtrl(env, TRAINED[n], n).evaluate(
        n_episodes=CFG.EVAL_EPISODES, seed=CFG.RANDOM_SEED + 12345)

# Hybrid
if "PPO-Tuned" in TRAINED:
    h_env = CampusEnvDiscrete(cluster_loads, priority_df,
                              seed=CFG.RANDOM_SEED + 1001)
    hyb = HybridController(h_env, TRAINED["PPO-Tuned"])
    print("Eval Hybrid-PPO+Greedy…")
    results["Hybrid-PPO+Greedy"] = hyb.evaluate(
        n_episodes=CFG.EVAL_EPISODES, seed=CFG.RANDOM_SEED + 12345)

summary = []
for name, m in results.items():
    summary.append({
        "Controller":      name,
        "Reward (μ)":      m["reward"].mean(),
        "Reward (σ)":      m["reward"].std(),
        "Energy Served":   m["served"].mean(),
        "Total Curtail":   m["curtail"].mean(),
        "Fairness (Jain)": m["fairness"].mean(),
        "Runtime (s)":     m["runtime"].mean(),
    })
summary_df = (pd.DataFrame(summary)
                .sort_values("Reward (μ)", ascending=False)
                .reset_index(drop=True))
summary_df.to_csv(f"{CFG.METRIC_PATH}/zoo_summary.csv", index=False)
summary_df.round(3)

## 13. Comparison plots (one figure per chart, ranked left → right)

Every figure below is an independent matplotlib figure — paste any of them
straight into a slide deck.

### 13.1 Mean reward by controller

In [ ]:
plt.figure(figsize=(13, 5.2))
order = summary_df["Controller"].tolist()
means = [results[n]["reward"].mean() for n in order]
errs  = [1.96 * results[n]["reward"].std() / np.sqrt(len(results[n]["reward"]))
         for n in order]
# colour rl variants vs baselines vs Greedy specially
def colour_for(n):
    if n == "Greedy":                       return "#2ca02c"
    if n in ["Random", "RoundRobin", "StaticPriority", "LowestPriorityFirst"]:
        return "#7f7f7f"
    if "PPO" in n:                          return "#d62728"
    if n in ["A2C", "TRPO"]:                return "#ff7f0e"
    if n in ["DQN", "QR-DQN"]:              return "#9467bd"
    if n in ["SAC", "TD3"]:                 return "#17becf"
    if "Hybrid" in n:                       return "#e377c2"
    return "#1f77b4"
cols = [colour_for(n) for n in order]
plt.bar(order, means, yerr=errs, color=cols, edgecolor="black", capsize=4)
plt.ylabel("Mean episode reward (95% CI)")
plt.title("Reward — every controller, ranked")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p10_reward_bars.png")
plt.show()

### 13.2 Energy served (kWh per episode)

In [ ]:
plt.figure(figsize=(13, 5.2))
served = [results[n]["served"].mean() for n in order]
plt.bar(order, served, color=cols, edgecolor="black")
plt.ylabel("Energy served (kWh / episode)")
plt.title("Energy delivered by each controller")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p11_served_bars.png")
plt.show()

### 13.3 Total curtailment

In [ ]:
plt.figure(figsize=(13, 5.2))
curt = [results[n]["curtail"].mean() for n in order]
plt.bar(order, curt, color=cols, edgecolor="black")
plt.ylabel("Curtailed energy (kWh / episode)")
plt.title("Total curtailment by controller")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p12_curtail_bars.png")
plt.show()

### 13.4 Jain fairness index

In [ ]:
plt.figure(figsize=(13, 5.2))
fair = [results[n]["fairness"].mean() for n in order]
plt.bar(order, fair, color=cols, edgecolor="black")
plt.ylim(0.85, 1.005)
plt.ylabel("Mean Jain index J")
plt.title("Fairness by controller (Jain index, higher = fairer)")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p13_fairness_bars.png")
plt.show()

### 13.5 Runtime per episode

In [ ]:
plt.figure(figsize=(13, 5.2))
rt = [results[n]["runtime"].mean() for n in order]
plt.bar(order, rt, color=cols, edgecolor="black")
plt.ylabel("Runtime (s / episode)")
plt.title("Wall-clock cost per controller (lower is better)")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p14_runtime_bars.png")
plt.show()

### 13.6 Reward distribution boxplot

In [ ]:
plt.figure(figsize=(13, 5.5))
data = [results[n]["reward"] for n in order]
bp = plt.boxplot(data, labels=order, patch_artist=True, showfliers=False)
for patch, c in zip(bp["boxes"], cols):
    patch.set_facecolor(c)
plt.ylabel("Episode reward")
plt.title("Per-controller reward distribution (60 episodes)")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p15_reward_box.png")
plt.show()

### 13.7 Best RL agent vs. Greedy — paired scatter

In [ ]:
rl_names = [n for n in order if n not in ["Random","RoundRobin","StaticPriority","LowestPriorityFirst","Greedy"]]
best_rl = max(rl_names, key=lambda n: results[n]["reward"].mean())

plt.figure(figsize=(7, 7))
greedy_r = results["Greedy"]["reward"]
best_r = results[best_rl]["reward"]
lo = min(greedy_r.min(), best_r.min())
hi = max(greedy_r.max(), best_r.max())
plt.scatter(greedy_r, best_r, alpha=0.7, color="navy")
plt.plot([lo, hi], [lo, hi], "k--", lw=1, label="y = x")
plt.xlabel("Greedy reward per episode")
plt.ylabel(f"{best_rl} reward per episode")
plt.title(f"{best_rl} vs. Greedy — paired per-episode reward")
plt.legend()
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p16_best_vs_greedy.png")
plt.show()

print(f"Best RL agent overall: {best_rl}")
print(f"  μ reward: {results[best_rl]['reward'].mean():.3f}")
print(f"  Greedy μ: {results['Greedy']['reward'].mean():.3f}")
print(f"  Δ      : {results[best_rl]['reward'].mean()-results['Greedy']['reward'].mean():+.3f}")

### 13.8 Reward of every RL variant against Greedy (Δ-bar)

In [ ]:
plt.figure(figsize=(11, 4.8))
delta = [results[n]["reward"].mean() - results["Greedy"]["reward"].mean()
         for n in rl_names]
sort_idx = np.argsort(delta)[::-1]
rl_sorted   = [rl_names[i] for i in sort_idx]
delta_sorted = [delta[i] for i in sort_idx]
col_sorted  = ["#2ca02c" if d > 0 else "#d62728" for d in delta_sorted]
plt.bar(rl_sorted, delta_sorted, color=col_sorted, edgecolor="black")
plt.axhline(0, color="black", lw=1)
plt.ylabel("Reward gap vs. Greedy")
plt.title("Every RL variant minus Greedy (positive = RL wins)")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p17_delta_vs_greedy.png")
plt.show()

## 14. Statistical significance — best RL vs Greedy (and vs every baseline)

In [ ]:
def cohens_d(x, y):
    diff = x - y
    return float(diff.mean() / (diff.std(ddof=1) + 1e-9))

def ci95(x):
    n = len(x); m = x.mean(); s = x.std(ddof=1)
    h = s * st.t.ppf(0.975, n - 1) / np.sqrt(n)
    return float(m - h), float(m + h)

best_rl_rewards = results[best_rl]["reward"]
rows = []
for name in [n for n in results if n != best_rl]:
    other = results[name]["reward"]
    if not np.any(best_rl_rewards - other):
        rows.append(dict(Baseline=name, Delta=0.0, CI="--",
                         t=np.nan, p=np.nan, d=0.0, BestWins="="))
        continue
    t_stat, t_p = st.ttest_rel(best_rl_rewards, other)
    lo, hi = ci95(best_rl_rewards - other)
    rows.append(dict(
        Baseline=name,
        Delta=float((best_rl_rewards - other).mean()),
        CI=f"[{lo:.2f}, {hi:.2f}]",
        t=float(t_stat), p=float(t_p),
        d=cohens_d(best_rl_rewards, other),
        BestWins="✓" if (best_rl_rewards - other).mean() > 0 and t_p < 0.05
                 else ("=" if t_p >= 0.05 else "✗"),
    ))
stat_df = pd.DataFrame(rows)
stat_df.to_csv(f"{CFG.METRIC_PATH}/zoo_stat_tests.csv", index=False)
stat_df.round(4)

## 15. Sensitivity per deficit regime — top-4 controllers

Compare the top RL variant + Greedy + PPO baseline + StaticPriority across
all five deficit regimes.  Individual plots per regime metric.

In [ ]:
top4 = []
for n in [best_rl, "Greedy", "PPO", "StaticPriority"]:
    if n in results and n not in top4: top4.append(n)
print("Evaluating per-regime sensitivity for:", top4)

sens_rows = []
for name in top4:
    for scen in PowerScenario.REG:
        if name in ["Random","RoundRobin","StaticPriority","LowestPriorityFirst","Greedy"]:
            env = CampusEnvDiscrete(cluster_loads, priority_df,
                                    scenario=scen, seed=CFG.RANDOM_SEED+99)
            ctrl_map = {"Random":RandomCtrl,"RoundRobin":RoundRobinCtrl,
                         "StaticPriority":StaticPriorityCtrl,
                         "LowestPriorityFirst":LowestPriorityFirstCtrl,
                         "Greedy":GreedyCtrl}
            ctrl = ctrl_map[name](env)
        elif name in ["SAC","TD3"]:
            env = CampusEnvContinuous(cluster_loads, priority_df,
                                      scenario=scen, seed=CFG.RANDOM_SEED+99)
            ctrl = RLCtrl(env, TRAINED[name], name)
        elif name == "MaskablePPO":
            env = CampusEnvMasked(cluster_loads, priority_df,
                                  scenario=scen, seed=CFG.RANDOM_SEED+99)
            ctrl = RLCtrl(env, TRAINED[name], name)
        elif name == "Hybrid-PPO+Greedy":
            env = CampusEnvDiscrete(cluster_loads, priority_df,
                                    scenario=scen, seed=CFG.RANDOM_SEED+99)
            ctrl = HybridController(env, TRAINED["PPO-Tuned"])
        else:
            env = CampusEnvDiscrete(cluster_loads, priority_df,
                                    scenario=scen, seed=CFG.RANDOM_SEED+99)
            ctrl = RLCtrl(env, TRAINED[name], name)
        m = ctrl.evaluate(n_episodes=15, seed=CFG.RANDOM_SEED+4242)
        sens_rows.append(dict(Controller=name, Scenario=scen,
                              Reward=m["reward"].mean(),
                              Served=m["served"].mean(),
                              Fairness=m["fairness"].mean()))
sens_df = pd.DataFrame(sens_rows)
sens_df.to_csv(f"{CFG.METRIC_PATH}/zoo_sensitivity.csv", index=False)
sens_df.round(3)

### 15.1 Reward by regime

In [ ]:
plt.figure(figsize=(10, 5))
pivot = sens_df.pivot(index="Scenario", columns="Controller", values="Reward")
pivot = pivot.reindex(list(PowerScenario.REG))
pivot.plot(kind="bar", ax=plt.gca(), edgecolor="black")
plt.ylabel("Mean episode reward")
plt.title("Reward by deficit regime — top 4 controllers")
plt.xticks(rotation=20)
plt.legend(loc="upper right")
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p18_sens_reward.png")
plt.show()

### 15.2 Energy served by regime

In [ ]:
plt.figure(figsize=(10, 5))
pivot = sens_df.pivot(index="Scenario", columns="Controller", values="Served")
pivot = pivot.reindex(list(PowerScenario.REG))
pivot.plot(kind="bar", ax=plt.gca(), edgecolor="black")
plt.ylabel("Energy served (kWh / episode)")
plt.title("Energy served by deficit regime")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p19_sens_served.png")
plt.show()

### 15.3 Fairness by regime

In [ ]:
plt.figure(figsize=(10, 5))
pivot = sens_df.pivot(index="Scenario", columns="Controller", values="Fairness")
pivot = pivot.reindex(list(PowerScenario.REG))
pivot.plot(kind="bar", ax=plt.gca(), edgecolor="black")
plt.ylim(0.85, 1.005)
plt.ylabel("Jain fairness index")
plt.title("Fairness by deficit regime")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p20_sens_fairness.png")
plt.show()

## 16. Reward-weight sensitivity heatmap (best RL agent)

In [ ]:
def eval_with_weights(w_critical, w_fairness, n_eps=10):
    env_cls = (CampusEnvMasked if best_rl == "MaskablePPO" else
               (CampusEnvContinuous if best_rl in ["SAC","TD3"] else
                CampusEnvDiscrete))
    env = env_cls(cluster_loads, priority_df,
                  reward_weights={"critical": w_critical,
                                   "fairness": w_fairness},
                  seed=CFG.RANDOM_SEED+50)
    return RLCtrl(env, TRAINED[best_rl], best_rl).evaluate(
        n_episodes=n_eps, seed=CFG.RANDOM_SEED+50)["reward"].mean()

w_crit_grid = [0.5, 1.0, 2.0, 4.0]
w_fair_grid = [0.0, 0.5, 1.0, 2.0]
heat = np.zeros((len(w_crit_grid), len(w_fair_grid)))
for i, wc in enumerate(w_crit_grid):
    for j, wf in enumerate(w_fair_grid):
        heat[i, j] = eval_with_weights(wc, wf, n_eps=8)

plt.figure(figsize=(8, 5.2))
sns.heatmap(heat, annot=True, fmt=".2f", cmap="magma",
            xticklabels=[f"wF={x}" for x in w_fair_grid],
            yticklabels=[f"wC={x}" for x in w_crit_grid],
            cbar_kws={"label": "Mean reward"})
plt.title(f"Reward sensitivity to (w_C, w_F) — {best_rl}")
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p21_weight_heatmap.png")
plt.show()

## 17. Per-cluster fairness deep-dive

For each top-4 controller we measure the served / demand ratio per cluster
across 30 episodes, then plot each cluster on its own figure (so the user can
read the spread cluster by cluster).

In [ ]:
def per_cluster_metrics(name, n_eps=30):
    if name in ["Random","RoundRobin","StaticPriority","LowestPriorityFirst","Greedy"]:
        env = CampusEnvDiscrete(cluster_loads, priority_df, seed=CFG.RANDOM_SEED+7777)
        ctrl = {"Random":RandomCtrl,"RoundRobin":RoundRobinCtrl,
                "StaticPriority":StaticPriorityCtrl,
                "LowestPriorityFirst":LowestPriorityFirstCtrl,
                "Greedy":GreedyCtrl}[name](env)
    elif name in ["SAC","TD3"]:
        env = CampusEnvContinuous(cluster_loads, priority_df, seed=CFG.RANDOM_SEED+7777)
        ctrl = RLCtrl(env, TRAINED[name], name)
    elif name == "MaskablePPO":
        env = CampusEnvMasked(cluster_loads, priority_df, seed=CFG.RANDOM_SEED+7777)
        ctrl = RLCtrl(env, TRAINED[name], name)
    elif name == "Hybrid-PPO+Greedy":
        env = CampusEnvDiscrete(cluster_loads, priority_df, seed=CFG.RANDOM_SEED+7777)
        ctrl = HybridController(env, TRAINED["PPO-Tuned"])
    else:
        env = CampusEnvDiscrete(cluster_loads, priority_df, seed=CFG.RANDOM_SEED+7777)
        ctrl = RLCtrl(env, TRAINED[name], name)
    out = np.zeros((n_eps, 3))
    for e in range(n_eps):
        obs, _ = env.reset(seed=CFG.RANDOM_SEED+7777+e)
        done = False
        while not done:
            a = ctrl.select_action(obs)
            obs, _, done, _, _ = env.step(a)
        d = np.array([h["demand"] for h in env._history]).sum(0)
        s = np.array([h["served"] for h in env._history]).sum(0)
        out[e] = s / np.maximum(d, 1e-9)
    return out

per_cluster_data = {n: per_cluster_metrics(n, n_eps=20) for n in top4}

### 17.1 Per-cluster served share — Cluster 0 (critical loads)

In [ ]:
plt.figure(figsize=(10, 4.6))
data = [per_cluster_data[n][:, 0] for n in top4]
plt.boxplot(data, labels=top4, patch_artist=True, showfliers=False)
plt.ylabel("Served / Demand — Cluster 0")
plt.title("Cluster 0 (critical) — served share by controller")
plt.ylim(0.5, 1.05)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p22_cluster0_share.png")
plt.show()

### 17.2 Per-cluster served share — Cluster 1 (academic/admin)

In [ ]:
plt.figure(figsize=(10, 4.6))
data = [per_cluster_data[n][:, 1] for n in top4]
plt.boxplot(data, labels=top4, patch_artist=True, showfliers=False)
plt.ylabel("Served / Demand — Cluster 1")
plt.title("Cluster 1 (academic/admin) — served share")
plt.ylim(0.4, 1.05)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p23_cluster1_share.png")
plt.show()

### 17.3 Per-cluster served share — Cluster 2 (residential/commercial)

In [ ]:
plt.figure(figsize=(10, 4.6))
data = [per_cluster_data[n][:, 2] for n in top4]
plt.boxplot(data, labels=top4, patch_artist=True, showfliers=False)
plt.ylabel("Served / Demand — Cluster 2")
plt.title("Cluster 2 (residential/commercial) — served share")
plt.ylim(0.2, 1.05)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p24_cluster2_share.png")
plt.show()

### 17.4 Radar — mean served share of top 4 across all clusters

In [ ]:
plt.figure(figsize=(7, 7))
ax = plt.subplot(111, polar=True)
theta = np.linspace(0, 2 * np.pi, 4)[:-1]
theta = np.append(theta, theta[0])
palette = ["#d62728", "#2ca02c", "#1f77b4", "#9467bd", "#ff7f0e"]
for name, c in zip(top4, palette):
    vals = per_cluster_data[name].mean(0)
    vals = np.append(vals, vals[0])
    ax.plot(theta, vals, lw=2, label=name, color=c)
    ax.fill(theta, vals, alpha=0.1, color=c)
ax.set_xticks(theta[:-1])
ax.set_xticklabels(["Cluster 0", "Cluster 1", "Cluster 2"])
ax.set_ylim(0, 1)
ax.set_title("Mean served share — top 4 controllers")
ax.legend(loc="lower right", bbox_to_anchor=(1.35, 0.0))
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p25_radar.png")
plt.show()

## 18. Learned policy heatmaps — best RL agent

For the winning RL agent we record (hour-of-day, deficit-bucket) → mean
curtailment per cluster.  Each cluster gets its own figure.

In [ ]:
def best_policy_table(n_eps=15):
    if best_rl == "MaskablePPO":
        env = CampusEnvMasked(cluster_loads, priority_df, seed=CFG.RANDOM_SEED+1000)
    elif best_rl in ["SAC","TD3"]:
        env = CampusEnvContinuous(cluster_loads, priority_df, seed=CFG.RANDOM_SEED+1000)
    else:
        env = CampusEnvDiscrete(cluster_loads, priority_df, seed=CFG.RANDOM_SEED+1000)
    rows = []
    for e in range(n_eps):
        obs, _ = env.reset(seed=CFG.RANDOM_SEED+1000+e)
        done = False
        while not done:
            a, _ = TRAINED[best_rl].predict(obs, deterministic=True)
            if isinstance(env, CampusEnvContinuous):
                curt = np.clip(np.asarray(a, dtype=np.float32), 0, 0.5)
            else:
                curt = env.decode_action(int(a))
            rows.append({"hour": env._hour_of_day(),
                          "deficit": float(obs[5]),
                          "c0": float(curt[0]), "c1": float(curt[1]), "c2": float(curt[2])})
            obs, _, done, _, _ = env.step(a if isinstance(env, CampusEnvContinuous) else int(a))
    pdf = pd.DataFrame(rows)
    pdf["bin"] = pd.cut(pdf["deficit"],
                        bins=[-0.01, 0.05, 0.20, 0.40, 0.60, 1.0],
                        labels=["~0%", "<20%", "<40%", "<60%", "≥60%"])
    return pdf

policy_df = best_policy_table(15)

### 18.1 Policy heatmap — Cluster 0

In [ ]:
plt.figure(figsize=(10, 4.6))
piv = (policy_df.groupby(["bin", "hour"])["c0"].mean()
                 .unstack(fill_value=0))
sns.heatmap(piv, cmap="magma", cbar_kws={"label": "mean curtailment"})
plt.title(f"{best_rl} learned policy — Cluster 0 (critical)")
plt.xlabel("Hour"); plt.ylabel("Deficit bucket")
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p26_policy_c0.png")
plt.show()

### 18.2 Policy heatmap — Cluster 1

In [ ]:
plt.figure(figsize=(10, 4.6))
piv = (policy_df.groupby(["bin", "hour"])["c1"].mean()
                 .unstack(fill_value=0))
sns.heatmap(piv, cmap="magma", cbar_kws={"label": "mean curtailment"})
plt.title(f"{best_rl} learned policy — Cluster 1 (academic/admin)")
plt.xlabel("Hour"); plt.ylabel("Deficit bucket")
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p27_policy_c1.png")
plt.show()

### 18.3 Policy heatmap — Cluster 2

In [ ]:
plt.figure(figsize=(10, 4.6))
piv = (policy_df.groupby(["bin", "hour"])["c2"].mean()
                 .unstack(fill_value=0))
sns.heatmap(piv, cmap="magma", cbar_kws={"label": "mean curtailment"})
plt.title(f"{best_rl} learned policy — Cluster 2 (residential/commercial)")
plt.xlabel("Hour"); plt.ylabel("Deficit bucket")
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p28_policy_c2.png")
plt.show()

## 19. Single-episode traces — best RL agent in action

A simple way to convince a reviewer (or yourself) that the learned policy
is doing something sensible: replay one episode and look at the timelines.

In [ ]:
def trace_episode(name, seed):
    if name in ["Random","RoundRobin","StaticPriority","LowestPriorityFirst","Greedy"]:
        env = CampusEnvDiscrete(cluster_loads, priority_df, seed=seed)
        ctrl = {"Random":RandomCtrl,"RoundRobin":RoundRobinCtrl,
                "StaticPriority":StaticPriorityCtrl,
                "LowestPriorityFirst":LowestPriorityFirstCtrl,
                "Greedy":GreedyCtrl}[name](env)
    elif name in ["SAC","TD3"]:
        env = CampusEnvContinuous(cluster_loads, priority_df, seed=seed)
        ctrl = RLCtrl(env, TRAINED[name], name)
    elif name == "MaskablePPO":
        env = CampusEnvMasked(cluster_loads, priority_df, seed=seed)
        ctrl = RLCtrl(env, TRAINED[name], name)
    elif name == "Hybrid-PPO+Greedy":
        env = CampusEnvDiscrete(cluster_loads, priority_df, seed=seed)
        ctrl = HybridController(env, TRAINED["PPO-Tuned"])
    else:
        env = CampusEnvDiscrete(cluster_loads, priority_df, seed=seed)
        ctrl = RLCtrl(env, TRAINED[name], name)
    obs, _ = env.reset(seed=seed)
    done = False
    while not done:
        a = ctrl.select_action(obs)
        obs, _, done, _, _ = env.step(a)
    h = env._history
    return (np.array([x["demand"] for x in h]),
            np.array([x["served"] for x in h]),
            np.array([x["curtailed"] for x in h]),
            np.array([x["available"] for x in h]),
            np.array([x["deficit"] for x in h]))

demand, served, curtailed, available, deficit = trace_episode(
    best_rl, CFG.RANDOM_SEED + 333)

### 19.1 Demand vs available power vs served — best agent

In [ ]:
plt.figure(figsize=(12, 4.5))
plt.fill_between(range(len(demand)), demand.sum(1), alpha=0.25,
                 color="steelblue", label="Total demand")
plt.plot(available, color="green", lw=1.8, label="Available power")
plt.plot(served.sum(1), color="darkred", lw=1.4, label="Served")
plt.ylabel("kW"); plt.xlabel("Hour")
plt.title(f"Demand vs supply — {best_rl} (single episode)")
plt.legend()
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p29_trace_supply.png")
plt.show()

### 19.2 Curtailment timeline by cluster — best agent

In [ ]:
plt.figure(figsize=(12, 4.5))
bot = np.zeros(len(curtailed))
for k, c in zip(range(3), ["#d62728", "#1f77b4", "#2ca02c"]):
    plt.bar(range(len(curtailed)), curtailed[:, k], bottom=bot,
            color=c, label=f"Cluster {k}", width=1.0)
    bot += curtailed[:, k]
plt.xlabel("Hour"); plt.ylabel("Curtailed kW")
plt.title(f"Curtailment timeline — {best_rl}")
plt.legend()
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p30_trace_curtail.png")
plt.show()

### 19.3 Deficit ratio trace

In [ ]:
plt.figure(figsize=(12, 3.6))
plt.plot(deficit, color="darkorange", lw=1.5)
plt.fill_between(range(len(deficit)), 0, deficit, alpha=0.25, color="darkorange")
plt.xlabel("Hour"); plt.ylabel("Deficit ratio ρ")
plt.title("Synthetic deficit signal during the traced episode")
plt.ylim(0, max(0.8, deficit.max() * 1.05))
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_PATH}/p31_trace_deficit.png")
plt.show()

## 20. Final summary

In [ ]:
best_overall = summary_df.iloc[0]
print("=" * 60)
print(f"BEST OVERALL CONTROLLER : {best_overall['Controller']}")
print(f"  mean reward : {best_overall['Reward (μ)']:.3f}")
print(f"  σ           : {best_overall['Reward (σ)']:.3f}")
print(f"  served kWh  : {best_overall['Energy Served']:.1f}")
print(f"  curtail kWh : {best_overall['Total Curtail']:.1f}")
print(f"  Jain J      : {best_overall['Fairness (Jain)']:.4f}")
print("=" * 60)
print("Top 5 of the zoo:")
print(summary_df.head(5).to_string(index=False))
print()
print("RL gap to Greedy:")
greedy_r = results["Greedy"]["reward"].mean()
for n in rl_names:
    delta = results[n]["reward"].mean() - greedy_r
    flag = "✓" if delta > 0 else ("≈" if abs(delta) < 0.5 else "✗")
    print(f"  {flag}  {n:25s}  Δ = {delta:+8.3f}")

## 21. Persist everything

All trained models, metrics CSVs and figures live under `./rl_zoo_artifacts/`.

In [ ]:
# Save trained RL models
for n, m in TRAINED.items():
    try:
        m.save(f"{CFG.MODEL_PATH}/{n}.zip")
    except Exception as e:
        print(f"  could not save {n}: {e}")

# Save logger histories
with open(f"{CFG.METRIC_PATH}/loggers.pkl", "wb") as f:
    pickle.dump({n: lg.episode_rewards for n, lg in LOGGERS.items()}, f)

# Save raw eval results
with open(f"{CFG.METRIC_PATH}/raw_results.pkl", "wb") as f:
    pickle.dump(results, f)

# Save config
with open(f"{CFG.SAVE_PATH}/config.json", "w") as f:
    json.dump(asdict(CFG), f, indent=2)

print("Artefacts written to", CFG.SAVE_PATH)
for p in sorted(Path(CFG.SAVE_PATH).rglob("*")):
    if p.is_file():
        print("  ", p)